# PlayTrain quickstart

Every environment is a single JavaScript file. A person can play it. An agent can train
on the same game.

This notebook installs PlayTrain, steps an environment, edits a game's source, measures
throughput, trains an agent, and plays a trained policy.

Set Runtime > Change runtime type > GPU before you start.

The numbers in this notebook are small. This VM has one physical core. The paper's
figures come from an 80-thread node. Each cell explains what its number means.

## 0. Install

This builds the native runtime from source. It takes about two minutes. Run it once.

In [ ]:
%%bash
set -euo pipefail

# clang for the runtime, cargo for the rasterizer. Colab has neither reliably.
apt-get -qq install -y clang >/dev/null
command -v cargo >/dev/null || {
  curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal --default-toolchain stable >/dev/null
}
export PATH="$HOME/.cargo/bin:$PATH"

# uv: the project's installer. Installing into Colab's system Python reuses the
# preinstalled torch instead of pulling down a fresh 2 GB copy.
command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh >/dev/null
export PATH="$HOME/.local/bin:$PATH"

# Into /content/src: a directory named `playtrain` next to the kernel's cwd
# shadows the installed package, and the trainers expect it as a sibling.
mkdir -p /content/src && cd /content/src
[ -d playtrain ]          || git clone -q --depth 1 https://github.com/heyodog0/playtrain.git
[ -d playtrain-trainers ] || git clone -q --depth 1 https://github.com/heyodog0/playtrain-trainers.git

# The native backend: rasterizer + QuickJS staticlibs, then the vectorized .so.
# (build_qjs.sh shallow-clones quickjs-ng and openlibm on first run.)
cd /content/src/playtrain
bash native/build_qjs.sh
bash native/build_qjs_vec.sh

# Not editable. An editable install leaves a bare `playtrain/` directory in
# site-packages and relies on a .pth hook to shadow it; when that hook does not
# run, `import playtrain` finds the bare directory and `playtrain.runtime` is
# missing. A regular install has no such failure mode.
uv pip install --system -q /content/src/playtrain
uv pip install --system -q /content/src/playtrain-trainers

# Fail here, not three cells later, if the kernel cannot see the package.
python3 -c "import playtrain.runtime, playtrain_trainers" || {
  echo "install did not land in the kernel's python:" >&2
  python3 -c "import sys; print(sys.executable); print(sys.path, file=sys.stderr)" >&2
  exit 1
}
echo "OK"

In [ ]:
import os, sys
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

import numpy as np, torch
from playtrain.runtime import GameEnv, NativeVecEnv, list_available_games

print(f"{len(list_available_games())} games:", ", ".join(sorted(list_available_games())[:12]), "...")
print("cpus:", os.cpu_count(), "| torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 1. An environment

This is a standard Gymnasium environment. It has `reset` and `step`. Observations are
`Box(0, 255, (64, 64, 3))`. Actions are `Discrete(8)`.

In [ ]:
import matplotlib.pyplot as plt

env = GameEnv(game="breakout", obs_size=64)
print("observation:", env.observation_space)
print("action:     ", env.action_space)

obs, info = env.reset(seed=0)
frames = [obs.copy()]
for _ in range(120):
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
    frames.append(obs.copy())

fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
for ax, i in zip(axes, np.linspace(0, len(frames) - 1, 6).astype(int)):
    ax.imshow(frames[i]); ax.set_title(f"t={i}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. The game is source you can read

Each game is about two hundred lines of p5-style JavaScript. The cell below prints the
source, changes one number, and runs the changed game.

In [ ]:
from pathlib import Path

src = Path("/content/src/playtrain/examples/games/js/breakout.js").read_text()
print(f"{len(src.splitlines())} lines\n")
print("\n".join(src.splitlines()[:45]))

In [ ]:
# Change one number and you have a new environment. Here: a much wider paddle,
# the `w: 80` in the `paddle = {...}` block printed above.
import re, tempfile

variant_src, n = re.subn(r"^(    w: )80,$", r"\g<1>200,", src, count=1, flags=re.M)
assert n == 1, "line not found - open the source above and pick a number to edit"

variant_path = Path(tempfile.mkdtemp()) / "breakout_widepaddle.js"
variant_path.write_text(variant_src)

def first_frames(game, n=90, seed=0):
    e = GameEnv(game=str(game), obs_size=64)
    o, _ = e.reset(seed=seed)
    for _ in range(n):
        o, *_ = e.step(e.action_space.sample())
    e.close()
    return o

fig, (a, b) = plt.subplots(1, 2, figsize=(6, 3))
a.imshow(first_frames("breakout"));   a.set_title("breakout");        a.axis("off")
b.imshow(first_frames(variant_path)); b.set_title("2.5x paddle width");  b.axis("off")
plt.tight_layout(); plt.show()

## 3. Speed

The first number is single-thread throughput. The second compares PlayTrain against ALE
on this same machine.

The ratio between them is the useful number. It holds on other hardware. The absolute
values do not, because this VM has one slow core.

In [ ]:
import time

def measure(num_envs, num_threads, steps=400, game="breakout"):
    venv = NativeVecEnv(game=game, num_envs=num_envs, num_threads=num_threads, obs_size=64)
    venv.reset(0)
    acts = np.zeros(num_envs, dtype=np.int64)
    for _ in range(20):                       # warmup
        venv.step(acts)
    t0 = time.perf_counter()
    for _ in range(steps):
        venv.step(acts)
    dt = time.perf_counter() - t0
    venv.close()
    return steps * num_envs / dt

!lscpu | egrep 'Model name|^CPU\\(s\\)|Thread|Core'

print(f"\\nsingle env, single thread: {measure(1, 1, steps=2000):,.0f} steps/s")


In [ ]:
# Same machine, same wall clock: PlayTrain vs ALE.
!uv pip install --system -q "ale-py>=0.10" autorom >/dev/null 2>&1
!python -c "import ale_py" && AutoROM --accept-license -q >/dev/null 2>&1 || true

import gymnasium as gym

def ale_sps(steps=2000):
    import ale_py; gym.register_envs(ale_py)
    e = gym.make("ALE/Breakout-v5", frameskip=1)
    e.reset(seed=0)
    for _ in range(50): e.step(0)
    t0 = time.perf_counter()
    for _ in range(steps):
        _, _, term, trunc, _ = e.step(e.action_space.sample())
        if term or trunc: e.reset()
    return steps / (time.perf_counter() - t0)

pt = measure(num_envs=1, num_threads=1, steps=2000)
try:
    ale = ale_sps()
    print(f"PlayTrain  {pt:>10,.0f} steps/s   (1 env, 1 thread)")
    print(f"ALE        {ale:>10,.0f} steps/s   (1 env)")
    print(f"ratio      {pt/ale:>10.2f}x")
except Exception as exc:
    print("ALE unavailable on this runtime:", exc)
    print(f"PlayTrain  {pt:,.0f} steps/s (1 env, 1 thread)")

The paper measures 80 threads and uses per-game AOT-compiled engine builds. Neither is
available here. Expect these numbers to be far below the published ones.

## 4. Train

This trains IMPALA (V-trace) on `breakout`.

Watch the throughput and the loss. Do not watch the reward. A 1.79M-step run on this VM
moved mean episode return from 87.5 to 95.5. A random policy scores about 79. The
checkpoint in step 5 took 100M steps on 60 environment threads.

This cell shows that the training loop works and what one step costs.

In [ ]:
import json, subprocess, threading

ncpu = os.cpu_count()
cfg = {
    "game": "breakout", "env_backend": "playtrain",
    "total_steps": 300_000,
    "batch_size": 32, "unroll_length": 64, "num_learner_threads": 1,
    "discounting": 0.99, "baseline_cost": 0.5, "entropy_cost": 0.01,
    "reward_clipping": "abs_one", "grad_norm_clipping": 40.0,
    "learning_rate": 0.0005, "seed": 0, "device": "auto",
    "obs_shape": [3, 64, 64], "num_actions": 8, "features_dim": 256,
    "net": "impala", "use_lstm": False, "frame_skip": 1,
    "inference_mode": "vec",
    "vec_workers": max(1, ncpu - 1), "vec_env_threads": 2, "vec_double_buffer": True,
    "stats_log_every": 20, "eval_every_steps": 0, "save_every_steps": 0,
    "resume": "off", "log_dir": "/content/outputs/impala_colab",
}
Path("/content/impala_colab.json").write_text(json.dumps(cfg, indent=1))
print(json.dumps(cfg, indent=1))

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/outputs/impala_colab/tb

In [ ]:
# Runs in the foreground so you can watch it; the TensorBoard panel above updates live.
# Stop early with the interrupt button - the run checkpoints as it goes.
!cd /content/src/playtrain-trainers && python -m playtrain_trainers.train_impala --config /content/impala_colab.json

## 5. Watch a trained policy

This checkpoint was trained for 100M steps on 12 workers with 5 environment threads
each. The file is 2.5 MB.

In [ ]:
CKPT_URL = "https://github.com/heyodog0/playtrain/releases/download/v0.1.0/impala_breakout_100M.pt"
ckpt_path = "/content/impala_breakout_100M.pt"

import urllib.request
if not Path(ckpt_path).exists():
    urllib.request.urlretrieve(CKPT_URL, ckpt_path)

from playtrain_trainers.impala.net import ImpalaNet

state = torch.load(ckpt_path, map_location="cpu", weights_only=False)["model_state_dict"]
net = ImpalaNet(observation_shape=(3, 64, 64), num_actions=8, features_dim=256,
                use_lstm=False, net="impala")
net.load_state_dict(state)
net.eval()

BLANK = {"reward": torch.zeros(1, 1), "done": torch.zeros(1, 1, dtype=torch.bool),
         "last_action": torch.zeros(1, 1, dtype=torch.int64)}

def act(obs):
    frame = torch.as_tensor(np.ascontiguousarray(obs.transpose(2, 0, 1)))[None, None]
    with torch.no_grad():
        out, _ = net({"frame": frame, **BLANK}, ())
    return int(out["policy_logits"][0, 0].argmax())

def episode(policy, seed, keep=False):
    e = GameEnv(game="breakout", obs_size=64)
    obs, _ = e.reset(seed=seed)
    total, frames = 0.0, [obs.copy()]
    for t in range(3000):
        obs, r, term, trunc, _ = e.step(policy(obs))
        total += r
        if keep: frames.append(obs.copy())
        if term or trunc: break
    e.close()
    return total, t + 1, frames

rng = np.random.default_rng(0)
rand_ret = [episode(lambda o: int(rng.integers(8)), s)[:2] for s in range(8)]
pol_ret  = [episode(act, s)[:2] for s in range(8)]

print(f"random   return {np.mean([r for r, _ in rand_ret]):7.1f}   episode length {np.mean([l for _, l in rand_ret]):6.0f}")
print(f"trained  return {np.mean([r for r, _ in pol_ret]):7.1f}   episode length {np.mean([l for _, l in pol_ret]):6.0f}")

In [ ]:
# Render it playing.
from PIL import Image
from IPython.display import Image as ShowImage, display

_, _, frames = episode(act, seed=0, keep=True)
gif = "/content/breakout_trained.gif"
imgs = [Image.fromarray(f).resize((256, 256), Image.NEAREST) for f in frames[::4]]
imgs[0].save(gif, save_all=True, append_images=imgs[1:], duration=40, loop=0)
display(ShowImage(filename=gif))

## Next

`examples/games/js/` holds the catalog. There is one file per environment.

`GAME_TEMPLATE.md` lists what a new game has to define.

Variants such as `frostbite.jungle.js` are edits of a base game. That is how the
generalization studies are built.

`NativeVecEnv` is the throughput path used above. `NativeVectorEnv` wraps it in the
Gymnasium `VectorEnv` API.